# B2 — Local Level Model

**Reference:** West & Harrison §2.3, §4.3; Petris §2.1

**Concepts introduced:**
- Signal-to-noise ratio and its effect on filtering
- RTS smoother: using future observations to refine past estimates
- Multi-step forecasting with widening uncertainty bands
- Log marginal likelihood and its role in model comparison

In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import matplotlib.pyplot as plt
from engine.models import make_local_level
from engine.filter import kalman_filter
from engine.smoother import rts_smoother
from engine.simulate import simulate
from engine.forecast import forecast_horizon

## 1. Local level model and signal-to-noise ratio

The local level model (W&H §4.3):

$$y_t = \theta_t + v_t, \quad v_t \sim N(0, V)$$
$$\theta_t = \theta_{t-1} + w_t, \quad w_t \sim N(0, W)$$

The **signal-to-noise ratio** $\kappa = W/V$ determines how responsive the filter is:
- $\kappa \to 0$: filter barely updates — trusts the model, ignores data
- $\kappa \to \infty$: filter follows $y_t$ closely — trusts data over model

In [ ]:
spec = make_local_level(V=2.0, W_level=0.5)
sim  = simulate(spec, n=100, seed=0)
fr   = kalman_filter(spec, sim.y)

t = np.arange(100)
std = np.sqrt(fr.C[:, 0, 0])

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

# Panel 1: observations + true state
axes[0].plot(t, sim.y[:, 0], ".", ms=4, alpha=0.5, label="obs y_t")
axes[0].plot(t, sim.theta_true[:, 0], "k-", lw=1.5, label="true θ_t")
axes[0].set_ylabel("value"); axes[0].legend()
axes[0].set_title("Local level (V=2.0, W=0.5, κ=0.25)")

# Panel 2: prior means (one-step predictions)
axes[1].plot(t, fr.a[:, 0], "C2-", lw=1.5, label="prior mean a_t")
axes[1].plot(t, sim.y[:, 0], ".", ms=3, alpha=0.3, color="grey")
axes[1].set_ylabel("value"); axes[1].legend()
axes[1].set_title("One-step-ahead predictions a_t")

# Panel 3: filtered means with uncertainty
axes[2].plot(t, sim.theta_true[:, 0], "k-", lw=1, label="true θ_t")
axes[2].plot(t, fr.m[:, 0], "C1-", lw=2, label="filtered mean m_t")
axes[2].fill_between(t, fr.m[:, 0]-1.96*std, fr.m[:, 0]+1.96*std,
                     alpha=0.25, color="C1", label="95% interval")
axes[2].set_xlabel("t"); axes[2].set_ylabel("value"); axes[2].legend()
axes[2].set_title("Filtered state")

plt.tight_layout()
plt.show()

## 2. RTS smoother

The **Rauch-Tung-Striebel (RTS) smoother** (W&H §4.5) runs a backward pass after
the filter to compute $p(\theta_t \mid y_{1:T})$ — using all observations, not just
those up to time $t$. The smoother is never worse than the filter.

**Backward recursion** (starting from $s_T = m_T$, $S_T = C_T$):

$$B_t = C_t G' R_{t+1}^{-1}$$
$$s_t = m_t + B_t(s_{t+1} - a_{t+1}), \quad S_t = C_t + B_t(S_{t+1} - R_{t+1})B_t'$$

The smoother mean $s_t$ uses future data to correct the filter. The smoother
covariance $S_t \leq C_t$ — always tighter than the filter.

In [ ]:
sr = rts_smoother(spec, fr)

std_s = np.sqrt(sr.S[:, 0, 0])
std_f = np.sqrt(fr.C[:, 0, 0])

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t, sim.theta_true[:, 0], "k-", lw=1, label="true θ_t")
ax.plot(t, fr.m[:, 0], "C1-", lw=1.5, alpha=0.7, label="filtered mean")
ax.fill_between(t, fr.m[:, 0]-1.96*std_f, fr.m[:, 0]+1.96*std_f,
                alpha=0.15, color="C1")
ax.plot(t, sr.s[:, 0], "C2-", lw=2, label="smoothed mean")
ax.fill_between(t, sr.s[:, 0]-1.96*std_s, sr.s[:, 0]+1.96*std_s,
                alpha=0.25, color="C2", label="95% smoother interval")
ax.set(xlabel="t", ylabel="value", title="Filter vs smoother")
ax.legend()
plt.tight_layout()
plt.show()

# Verify: smoother at T equals filter at T
print(f"s[T-1] = {sr.s[-1, 0]:.6f}  (smoother)")
print(f"m[T-1] = {fr.m[-1, 0]:.6f}  (filter)")
print("Smoother mean std (avg):", np.sqrt(sr.S[:, 0, 0]).mean().round(4))
print("Filter   mean std (avg):", np.sqrt(fr.C[:, 0, 0]).mean().round(4))

## 3. Multi-step forecast

The one-step-ahead predictive distribution at time $t$ is (W&H §4.4):

$$p(y_{t+1} \mid y_{1:t}) = N(F\, a_{t+1},\, Q_{t+1})$$

For $h$-step forecasting we apply the state equation $h$ times, propagating
uncertainty: each step adds $W$ to the state covariance and projects through $F$.
The uncertainty widens monotonically for a local level (random walk).

In [ ]:
fc = forecast_horizon(spec, fr, h=20)

t_fc = np.arange(100, 120)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t, sim.y[:, 0], ".", ms=3, alpha=0.5, color="grey", label="obs (in-sample)")
ax.plot(t, fr.m[:, 0], "C1-", lw=1.5, label="filtered mean")
ax.plot(t_fc, fc.means[:, 0], "C3-", lw=2, label="forecast mean")
ax.fill_between(t_fc, fc.lower[:, 0], fc.upper[:, 0],
                alpha=0.3, color="C3", label="95% forecast interval")
ax.axvline(99.5, color="k", ls="--", lw=1)
ax.set(xlabel="t", ylabel="value", title="20-step forecast — local level")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Log marginal likelihood

The Kalman filter computes the **log marginal likelihood** as a by-product:

$$\log p(y_{1:T}) = \sum_{t=1}^T \log N(e_t;\, 0, Q_t)
= -\frac{T}{2}\log(2\pi) - \frac{1}{2}\sum_t \left[\log|Q_t| + e_t^\top Q_t^{-1} e_t\right]$$

This is the probability of the data under the model. It is the gold standard for
comparing models with different structures (no penalty term needed — complexity
is already penalised by integrating out $\theta_{1:T}$).

In [ ]:
# Compare log-likelihoods for different V values
V_grid = [0.5, 1.0, 2.0, 4.0, 8.0]
logliks = []
for V_try in V_grid:
    sp = make_local_level(V=V_try, W_level=0.5)
    logliks.append(kalman_filter(sp, sim.y).loglik)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(V_grid, logliks, "o-", lw=2)
ax.axvline(2.0, color="r", ls="--", label="true V=2.0")
ax.set(xlabel="V (observation noise)", ylabel="log p(y | V, W=0.5)",
       title="Likelihood profile over V (true W=0.5 fixed)")
ax.legend()
plt.tight_layout()
plt.show()
print("Best V from grid:", V_grid[int(np.argmax(logliks))])

## Exercises

**Exercise 1** — Fix W=0.5. Run the filter for V ∈ {0.5, 2.0, 8.0} and plot the
three filtered means on one axis alongside the true state. What does large V do
to the filter?

**Exercise 2** — Verify that `sr.s[-1, 0] == fr.m[-1, 0]` (within floating-point
tolerance). Explain in one sentence why this must be true from the RTS equations.

**Exercise 3** — Compute the log-likelihood for W ∈ {0.1, 0.5, 1.0, 2.0} with
V=2.0 fixed. Plot it. At which W is it maximized?